# Validation Notebook: CLT + FEA Comparison

**Laminate:** [0/45/−45/90]s (IM7/8552, 8 plies, t = 1.0 mm)  
**Plate:** 300 × 300 mm SSSS  
**Load:** q = 1 000 Pa uniform transverse pressure  

**Cells:**
1. CLT: ABD matrix + per-ply table (strains, Tsai-Wu, Hashin FT/FC/MT/MC)
2. CalculiX: run FEA solver via subprocess
3. CLT vs FEA comparison table
4. Bar chart: CLT vs FEA deflection + σₓₓ with % error
5. Bar chart: Tsai-Wu vs Hashin failure indices per ply

In [ ]:
# ── Cell 1: CLT — ABD matrix + per-ply failure indices ────────────────────────
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import load_materials
from clt import evaluate_laminate, navier_center_deflection

# ── material & laminate definition ────────────────────────────────────────────
# load_materials() strips strength cols → read raw CSV for strengths
mat_norm = load_materials(ROOT / 'data' / 'materials.csv').iloc[0]
mat_raw  = pd.read_csv(ROOT / 'data' / 'materials.csv').iloc[0]

E1    = float(mat_norm['E1'])
E2    = float(mat_norm['E2'])
G12   = float(mat_norm['G12'])
nu12  = float(mat_norm['v12'])
t_ply = float(mat_norm['t_ply'])

strengths = {
    'X_T': float(mat_raw['S1T_Pa']),
    'X_C': float(mat_raw['S1C_Pa']),
    'Y_T': float(mat_raw['S2T_Pa']),
    'Y_C': float(mat_raw['S2C_Pa']),
    'S12': float(mat_raw['S12_Pa']),
}

BASE_STACK = [0, 45, -45, 90, 90, -45, 45, 0]
LX, LY, Q_LOAD = 0.30, 0.30, 1_000.0

N = np.zeros(3)
M = np.array([Q_LOAD * LX**2 / 8.0, 0.0, 0.0])   # representative bending moment

res = evaluate_laminate(E1, E2, G12, nu12, BASE_STACK, t_ply, N, M,
                        strengths=strengths)

# ── ABD matrix display ────────────────────────────────────────────────────────
labels = ['x', 'y', 'xy']
A, B, D = res['A'], res['B'], res['D']

print('=' * 60)
print('  A matrix  [N/m]')
df_A = pd.DataFrame(A, index=labels, columns=labels)
display(df_A.style.format('{:.4e}'))

print('  B matrix  [N]')
df_B = pd.DataFrame(B, index=labels, columns=labels)
display(df_B.style.format('{:.4e}'))

print('  D matrix  [N·m]')
df_D = pd.DataFrame(D, index=labels, columns=labels)
display(df_D.style.format('{:.4e}'))

print(f"\n||B|| = {np.linalg.norm(B):.4e}  (near-zero → symmetric laminate ✓)")

# ── per-ply table ─────────────────────────────────────────────────────────────
rows = []
for p in res['plies']:
    e12b = p['eps_bot_12']
    rows.append({
        'Ply': p['k'] + 1,
        'θ (°)': p['theta_deg'],
        'z_bot (mm)': round(p['z_bot'] * 1e3, 4),
        'z_top (mm)': round(p['z_top'] * 1e3, 4),
        'ε₁_bot (µε)': round(e12b[0] * 1e6, 2),
        'ε₂_bot (µε)': round(e12b[1] * 1e6, 2),
        'γ₁₂_bot (µε)': round(e12b[2] * 1e6, 2),
        'Tsai-Wu_bot': round(p.get('tsai_wu_bot', float('nan')), 6),
        'FT_bot': round(p.get('hashin_FT_bot', float('nan')), 6),
        'FC_bot': round(p.get('hashin_FC_bot', float('nan')), 6),
        'MT_bot': round(p.get('hashin_MT_bot', float('nan')), 6),
        'MC_bot': round(p.get('hashin_MC_bot', float('nan')), 6),
    })

df_ply = pd.DataFrame(rows)
print('\n  Per-ply results (bottom face)')
display(df_ply)

In [ ]:
# ── Cell 2: CalculiX FEA run via subprocess ────────────────────────────────────
import subprocess, shutil

FEA_DIR   = ROOT / 'fea'
INP_FILE  = FEA_DIR / 'abaqus_inputs' / 'plate_ss.inp'
CCX_NAMES = ['ccx', 'ccx_2.21', 'ccx_2.20', 'ccx_2.19']

ccx_bin = next((shutil.which(c) for c in CCX_NAMES if shutil.which(c)), None)

if ccx_bin and INP_FILE.exists():
    inp_stem = INP_FILE.stem
    print(f'Running CalculiX: {ccx_bin} {inp_stem}')
    result = subprocess.run(
        [ccx_bin, inp_stem],
        cwd=str(INP_FILE.parent),
        capture_output=True,
        text=True,
        timeout=120,
    )
    print('--- stdout ---')
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode == 0:
        print(f'\nCalculiX exited cleanly (rc={result.returncode}) ✓')
    else:
        print(f'\nCalculiX rc={result.returncode} — check stderr below')
        print(result.stderr[-1000:])
else:
    print('CalculiX binary not found on PATH — using pre-computed fea_summary.csv.')
    print('To install: conda install -c conda-forge calculix')
    print('FEA results will be loaded from:', ROOT / 'fea' / 'results' / 'fea_summary.csv')

In [ ]:
# ── Cell 3: CLT vs FEA comparison table ───────────────────────────────────────
from compare_clt_fea import compute_clt_values, load_fea_values, compute_errors

clt_vals = compute_clt_values()
fea_vals = load_fea_values()
rows_cmp = compute_errors(clt_vals, fea_vals)
df_cmp   = pd.DataFrame(rows_cmp)

print('\n  CLT vs FEA Comparison')
print('=' * 60)
display(df_cmp)

threshold = 5.0
all_pass  = (df_cmp['Err_%'] < threshold).all()
print(f'\nAll errors < {threshold}%: {"PASS ✓" if all_pass else "FAIL ✗"}')

In [ ]:
# ── Cell 4: Bar chart — CLT vs FEA deflection + σₓₓ with % error ──────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

metrics = [
    ('Deflection', 'mm',  clt_vals['deflection_m'] * 1e3,    fea_vals['deflection_m'] * 1e3),
    ('σₓₓ (peak)', 'MPa', clt_vals['sigma_xx_max_Pa'] * 1e-6, fea_vals['sigma_xx_max_Pa'] * 1e-6),
]

for ax, (label, unit, clt_v, fea_v) in zip(axes, metrics):
    bars = ax.bar(['CLT', 'FEA'], [clt_v, fea_v],
                  color=['steelblue', 'darkorange'], width=0.4, edgecolor='k')
    err_pct = abs(clt_v - fea_v) / abs(fea_v) * 100
    for bar, val in zip(bars, [clt_v, fea_v]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'{label} [{unit}]\nError = {err_pct:.2f}%', fontsize=10)
    ax.set_ylabel(unit)
    ax.set_ylim(0, max(clt_v, fea_v) * 1.2)

fig.suptitle('CLT vs FEA Comparison — Baseline [0/45/−45/90]s', fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(ROOT / 'figures' / 'fea_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/fea_comparison.png')

In [ ]:
# ── Cell 5: Failure index bar chart — Tsai-Wu vs Hashin per ply ───────────────
ply_labels = [f"Ply {p['k']+1}\n{p['theta_deg']}°" for p in res['plies']]
n = len(res['plies'])
x = np.arange(n)

fi_keys = ['tsai_wu_bot', 'hashin_FT_bot', 'hashin_FC_bot', 'hashin_MT_bot', 'hashin_MC_bot']
fi_labels = ['Tsai-Wu', 'Hashin FT', 'Hashin FC', 'Hashin MT', 'Hashin MC']
colors = ['#2196F3', '#4CAF50', '#F44336', '#FF9800', '#9C27B0']

width = 0.15
offsets = np.linspace(-2, 2, len(fi_keys)) * width

fig, ax = plt.subplots(figsize=(12, 5))
for offset, key, lbl, color in zip(offsets, fi_keys, fi_labels, colors):
    vals = [p.get(key, 0.0) for p in res['plies']]
    ax.bar(x + offset, vals, width=width, label=lbl, color=color, edgecolor='k', linewidth=0.5)

ax.axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='Failure (FI = 1.0)')
ax.set_xticks(x)
ax.set_xticklabels(ply_labels, fontsize=9)
ax.set_ylabel('Failure Index')
ax.set_title('Failure Indices per Ply — Tsai-Wu vs Hashin (FT/FC/MT/MC)\n'
             '[0/45/−45/90]s under q = 1 kPa')
ax.legend(loc='upper right', fontsize=8)
fig.tight_layout()
fig.savefig(ROOT / 'figures' / 'failure_indices.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/failure_indices.png')

print('\n✅ CHECKPOINT 2: Notebook complete — all 5 cells executed.')